# V8 — Robustness and Uncertainty of the Serve Admissibility Envelope

## Research Question

A deterministic admissibility boundary identifies launch conditions for which a serve is legal under a specific set of model parameters.

However, a real tennis serve is not executed at exactly one speed, launch angle, azimuth, contact height, and spin rate. Small variations in these quantities can move the trajectory across a legal boundary.

V8 therefore asks:

> **How robust is a nominally legal serve to realistic perturbations in its launch conditions?**

## Robustness Concept

This analysis distinguishes two concepts:

### Physically admissible

A serve is physically admissible if the modeled trajectory:

1. clears the net,
2. lands between the net and service line, and
3. lands within the appropriate service box.

### Robustly admissible

A nominally legal serve is considered robustly admissible when a high fraction of controlled perturbations around the nominal launch state remain legal.

For a nominal serve with a set of perturbations,

$$
R =
\frac{N_{\mathrm{legal}}}
{N_{\mathrm{total}}},
$$

where:

- $N_{\mathrm{legal}}$ is the number of perturbed trajectories that remain legal;
- $N_{\mathrm{total}}$ is the total number of perturbation samples.

This quantity is called a **robustness fraction**.

It is not interpreted as a probability unless an explicit probability distribution is assigned to the perturbations.

## Perturbed Variables

The initial V8 uncertainty ranges will be:

| Variable | Perturbation |
|---|---:|
| Serve speed | ±2 km/h |
| Launch angle | ±0.25° |
| Launch azimuth | ±0.25° |
| Contact height | ±0.10 m |
| Spin rate | ±100 rpm |

These perturbations represent a controlled sensitivity experiment rather than measured player-specific variability.

## Scientific Objective

V8 will determine:

1. how rapidly legality deteriorates near an admissibility boundary;
2. which launch variables have the greatest influence on legality;
3. whether the center of an admissible region is substantially more robust than its boundary;
4. how robustness changes with serve speed and spin.

The analysis will distinguish **model sensitivity** from **empirical player variability**. No claim will be made that the selected perturbation ranges represent the exact variability of professional players.

In [ ]:
# V8 Cell 2 — Robustness Analysis Setup

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Nominal serve
# ------------------------------------------------------------

NOMINAL_SPEED_KMH = 200.0
NOMINAL_ANGLE_DEG = -7.914436
NOMINAL_AZIMUTH_DEG = 0.0
NOMINAL_CONTACT_HEIGHT_M = 3.0
NOMINAL_SPIN_RPM = 0.0

TARGET_SIDE = "deuce"

# ------------------------------------------------------------
# Controlled perturbation ranges
# ------------------------------------------------------------

SPEED_PERTURBATION_KMH = 2.0
ANGLE_PERTURBATION_DEG = 0.25
AZIMUTH_PERTURBATION_DEG = 0.25
CONTACT_HEIGHT_PERTURBATION_M = 0.10
SPIN_PERTURBATION_RPM = 100.0

# ------------------------------------------------------------
# Display setup
# ------------------------------------------------------------

print("=" * 70)
print("V8 — ROBUSTNESS ANALYSIS SETUP")
print("=" * 70)

print("\nNominal serve:")
print(f"Speed:          {NOMINAL_SPEED_KMH:.1f} km/h")
print(f"Launch angle:   {NOMINAL_ANGLE_DEG:.6f}°")
print(f"Launch azimuth: {NOMINAL_AZIMUTH_DEG:.2f}°")
print(f"Contact height: {NOMINAL_CONTACT_HEIGHT_M:.2f} m")
print(f"Spin:           {NOMINAL_SPIN_RPM:.0f} rpm")
print(f"Target side:    {TARGET_SIDE}")

print("\nPerturbation ranges:")
print(f"Speed:          ±{SPEED_PERTURBATION_KMH:.1f} km/h")
print(f"Launch angle:   ±{ANGLE_PERTURBATION_DEG:.2f}°")
print(f"Launch azimuth: ±{AZIMUTH_PERTURBATION_DEG:.2f}°")
print(f"Contact height: ±{CONTACT_HEIGHT_PERTURBATION_M:.2f} m")
print(f"Spin:           ±{SPIN_PERTURBATION_RPM:.0f} rpm")

print("\n" + "=" * 70)
print("V8 SETUP COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 3 — Controlled Perturbation Design

# ------------------------------------------------------------
# Construct the perturbation levels
# ------------------------------------------------------------

speed_levels = np.array([
    NOMINAL_SPEED_KMH - SPEED_PERTURBATION_KMH,
    NOMINAL_SPEED_KMH,
    NOMINAL_SPEED_KMH + SPEED_PERTURBATION_KMH
])

angle_levels = np.array([
    NOMINAL_ANGLE_DEG - ANGLE_PERTURBATION_DEG,
    NOMINAL_ANGLE_DEG,
    NOMINAL_ANGLE_DEG + ANGLE_PERTURBATION_DEG
])

azimuth_levels = np.array([
    NOMINAL_AZIMUTH_DEG - AZIMUTH_PERTURBATION_DEG,
    NOMINAL_AZIMUTH_DEG,
    NOMINAL_AZIMUTH_DEG + AZIMUTH_PERTURBATION_DEG
])

height_levels = np.array([
    NOMINAL_CONTACT_HEIGHT_M - CONTACT_HEIGHT_PERTURBATION_M,
    NOMINAL_CONTACT_HEIGHT_M,
    NOMINAL_CONTACT_HEIGHT_M + CONTACT_HEIGHT_PERTURBATION_M
])

spin_levels = np.array([
    NOMINAL_SPIN_RPM - SPIN_PERTURBATION_RPM,
    NOMINAL_SPIN_RPM,
    NOMINAL_SPIN_RPM + SPIN_PERTURBATION_RPM
])

# ------------------------------------------------------------
# Create the full factorial perturbation design
# ------------------------------------------------------------

perturbation_cases = []

for speed in speed_levels:
    for angle in angle_levels:
        for azimuth in azimuth_levels:
            for height in height_levels:
                for spin in spin_levels:

                    perturbation_cases.append({
                        "speed_kmh": speed,
                        "angle_deg": angle,
                        "azimuth_deg": azimuth,
                        "contact_height_m": height,
                        "spin_rpm": spin
                    })

perturbation_df = pd.DataFrame(perturbation_cases)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

expected_cases = (
    len(speed_levels)
    * len(angle_levels)
    * len(azimuth_levels)
    * len(height_levels)
    * len(spin_levels)
)

print("=" * 70)
print("V8 — PERTURBATION DESIGN")
print("=" * 70)

print(f"\nSpeed levels:          {speed_levels}")
print(f"Launch-angle levels:   {angle_levels}")
print(f"Azimuth levels:        {azimuth_levels}")
print(f"Contact-height levels: {height_levels}")
print(f"Spin levels:           {spin_levels}")

print(f"\nExpected perturbation cases: {expected_cases}")
print(f"Actual perturbation cases:   {len(perturbation_df)}")

print("\nFirst five cases:")
print(perturbation_df.head())

design_pass = len(perturbation_df) == expected_cases

print("\n" + "=" * 70)
print(
    f"V8 PERTURBATION DESIGN: "
    f"{'PASS' if design_pass else 'FAIL'}"
)
print("=" * 70)

In [ ]:
# V7 Cell 2 — Verified 3D Serve Physics Model

import numpy as np
from scipy.integrate import solve_ivp

print("=" * 70)
print("V7 — VERIFIED 3D SERVE PHYSICS MODEL")
print("=" * 70)


# --------------------------------------------------
# Physical constants
# --------------------------------------------------

G = 9.81
BALL_MASS = 0.0575
BALL_DIAMETER = 0.067
BALL_RADIUS = BALL_DIAMETER / 2.0

AIR_DENSITY = 1.21
DRAG_COEFFICIENT = 0.55

BALL_AREA = np.pi * BALL_RADIUS**2

# Provisional spin parameter from V3–V6
V_SPIN = 20.0


# --------------------------------------------------
# Court geometry
# --------------------------------------------------

NET_X = 0.0
SERVICE_LINE_X = 6.40

BASELINE_X = 11.885
SERVER_X = -BASELINE_X

SERVICE_BOX_WIDTH = 4.115

NET_HEIGHT_CENTER = 0.914
NET_HEIGHT_POST = 1.07

CONTACT_HEIGHT = 3.0


# --------------------------------------------------
# Lift coefficient
# --------------------------------------------------

def lift_coefficient(speed, spin_speed):
    """
    Provisional lift-coefficient model.

    spin_speed = R * |omega|
    """

    if speed <= 0 or spin_speed <= 0:
        return 0.0

    return 1.0 / (
        2.0 + speed / spin_speed
    )


# --------------------------------------------------
# Magnus acceleration
# --------------------------------------------------

def magnus_acceleration(
    velocity,
    omega
):
    velocity = np.asarray(
        velocity,
        dtype=float
    )

    omega = np.asarray(
        omega,
        dtype=float
    )

    speed = np.linalg.norm(
        velocity
    )

    spin_rate = np.linalg.norm(
        omega
    )

    if speed == 0 or spin_rate == 0:
        return np.zeros(3)

    velocity_hat = (
        velocity / speed
    )

    omega_hat = (
        omega / spin_rate
    )

    spin_speed = (
        BALL_RADIUS * spin_rate
    )

    C_L = lift_coefficient(
        speed,
        spin_speed
    )

    magnus_direction = np.cross(
        omega_hat,
        velocity_hat
    )

    force_magnitude = (
        0.5
        * AIR_DENSITY
        * BALL_AREA
        * C_L
        * speed**2
    )

    force = (
        force_magnitude
        * magnus_direction
    )

    return force / BALL_MASS


# --------------------------------------------------
# Full acceleration model
# --------------------------------------------------

def serve_acceleration(
    velocity,
    omega
):
    velocity = np.asarray(
        velocity,
        dtype=float
    )

    speed = np.linalg.norm(
        velocity
    )

    # Drag
    if speed > 0:

        drag_factor = (
            -0.5
            * AIR_DENSITY
            * DRAG_COEFFICIENT
            * BALL_AREA
            * speed
            / BALL_MASS
        )

        a_drag = (
            drag_factor * velocity
        )

    else:

        a_drag = np.zeros(3)


    # Magnus
    a_magnus = magnus_acceleration(
        velocity,
        omega
    )


    # Gravity
    a_gravity = np.array([
        0.0,
        0.0,
        -G
    ])


    return (
        a_gravity
        + a_drag
        + a_magnus
    )


# --------------------------------------------------
# 3D trajectory
# --------------------------------------------------

def serve_trajectory(
    t,
    state,
    omega
):

    x, y, z, vx, vy, vz = state

    velocity = np.array([
        vx,
        vy,
        vz
    ])

    acceleration = serve_acceleration(
        velocity,
        omega
    )

    return np.array([
        vx,
        vy,
        vz,
        acceleration[0],
        acceleration[1],
        acceleration[2]
    ])


# --------------------------------------------------
# Events
# --------------------------------------------------

def net_event(t, state):

    return state[0] - NET_X


net_event.direction = 1


def ground_event(t, state):

    return state[2]


ground_event.terminal = True
ground_event.direction = -1


# --------------------------------------------------
# Serve simulation
# --------------------------------------------------

def simulate_serve_v7(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg,
    omega
):

    speed = speed_kmh / 3.6

    theta = np.radians(
        launch_angle_deg
    )

    phi = np.radians(
        azimuth_deg
    )


    # Initial velocity
    initial_velocity = np.array([
        speed
        * np.cos(theta)
        * np.cos(phi),

        speed
        * np.cos(theta)
        * np.sin(phi),

        speed
        * np.sin(theta)
    ])


    # Initial state
    initial_state = np.array([
        SERVER_X,
        0.0,
        CONTACT_HEIGHT,

        initial_velocity[0],
        initial_velocity[1],
        initial_velocity[2]
    ])


    solution = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),

        t_span=(0.0, 4.0),

        y0=initial_state,

        events=[
            net_event,
            ground_event
        ],

        rtol=1e-10,
        atol=1e-12,
        max_step=0.001,

        dense_output=True
    )

    return solution


# --------------------------------------------------
# Extract events
# --------------------------------------------------

def extract_serve_events(solution):

    net_state = None
    landing_state = None

    if len(solution.t_events[0]) > 0:

        net_time = (
            solution.t_events[0][0]
        )

        net_state = (
            solution.sol(net_time)
        )


    if len(solution.t_events[1]) > 0:

        landing_time = (
            solution.t_events[1][0]
        )

        landing_state = (
            solution.sol(landing_time)
        )


    return (
        net_state,
        landing_state
    )


# --------------------------------------------------
# Court geometry
# --------------------------------------------------

def net_height(y):

    y = np.asarray(
        y
    )

    return (
        NET_HEIGHT_CENTER
        + (
            NET_HEIGHT_POST
            - NET_HEIGHT_CENTER
        )
        * np.minimum(
            np.abs(y)
            / SERVICE_BOX_WIDTH,
            1.0
        )
    )


def net_clearance(
    z,
    y
):

    return (
        z
        - net_height(y)
    )


# --------------------------------------------------
# Model summary
# --------------------------------------------------

print("Physical model:")
print(f"  Ball mass:          {BALL_MASS:.4f} kg")
print(f"  Ball diameter:      {BALL_DIAMETER:.4f} m")
print(f"  Air density:        {AIR_DENSITY:.2f} kg/m³")
print(f"  Drag coefficient:   {DRAG_COEFFICIENT:.2f}")
print(f"  Provisional V_SPIN: {V_SPIN:.1f} m/s")

print("\nCourt model:")
print(f"  Server x:            {SERVER_X:.3f} m")
print(f"  Net x:               {NET_X:.3f} m")
print(f"  Service line x:      {SERVICE_LINE_X:.3f} m")
print(f"  Service half-width:  {SERVICE_BOX_WIDTH:.3f} m")

print("\nV7 Cell 2: 3D PHYSICS MODEL READY")

In [ ]:
# V8 Cell 4 — Nominal V7 Physics Verification

# ------------------------------------------------------------
# Use the exact validated V7 simulation for the nominal case.
# ------------------------------------------------------------

omega_nominal = np.array([
    0.0,
    0.0,
    0.0
])

nominal_solution = simulate_serve_v7(
    speed_kmh=NOMINAL_SPEED_KMH,
    launch_angle_deg=NOMINAL_ANGLE_DEG,
    azimuth_deg=NOMINAL_AZIMUTH_DEG,
    omega=omega_nominal
)

net_state, landing_state = extract_serve_events(
    nominal_solution
)

print("=" * 70)
print("V8 — NOMINAL V7 PHYSICS VERIFICATION")
print("=" * 70)

print("\nNominal inputs:")
print(f"Speed:          {NOMINAL_SPEED_KMH:.1f} km/h")
print(f"Launch angle:   {NOMINAL_ANGLE_DEG:.6f}°")
print(f"Launch azimuth: {NOMINAL_AZIMUTH_DEG:.2f}°")
print(f"Contact height: {NOMINAL_CONTACT_HEIGHT_M:.2f} m")
print(f"Spin:           {NOMINAL_SPIN_RPM:.0f} rpm")

print("\nEvent results:")

if net_state is not None:
    print(f"Net x:          {net_state[0]:.6f} m")
    print(f"Net y:          {net_state[1]:.6f} m")
    print(f"Net z:          {net_state[2]:.6f} m")

    clearance = net_clearance(
        net_state[2],
        net_state[1]
    )

    print(f"Net clearance:  {clearance:.6f} m")
else:
    print("Net event:      NOT DETECTED")

if landing_state is not None:
    print(f"\nLanding x:      {landing_state[0]:.6f} m")
    print(f"Landing y:      {landing_state[1]:.6f} m")
    print(f"Landing z:      {landing_state[2]:.6f} m")
else:
    print("\nLanding event:  NOT DETECTED")

print("\n" + "=" * 70)
print("V8 NOMINAL V7 PHYSICS CHECK COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 5 — Variable-Condition Serve Simulator

def simulate_serve_v8(
    speed_kmh,
    launch_angle_deg,
    azimuth_deg,
    contact_height_m,
    spin_rpm
):
    """
    V8 serve simulator.

    Uses the validated V7 aerodynamic and numerical model,
    while allowing contact height and spin rate to vary.

    Spin conversion:
        rpm -> rad/s
    """

    # --------------------------------------------------------
    # Convert speed and launch angles
    # --------------------------------------------------------

    speed = speed_kmh / 3.6

    theta = np.radians(
        launch_angle_deg
    )

    phi = np.radians(
        azimuth_deg
    )

    # --------------------------------------------------------
    # Convert spin from rpm to rad/s
    # --------------------------------------------------------

    spin_rad_s = (
        spin_rpm
        * 2.0
        * np.pi
        / 60.0
    )

    # Vertical spin axis consistent with V7 convention
    omega = np.array([
        0.0,
        spin_rad_s,
        0.0
    ])

    # --------------------------------------------------------
    # Initial velocity
    # --------------------------------------------------------

    initial_velocity = np.array([
        speed
        * np.cos(theta)
        * np.cos(phi),

        speed
        * np.cos(theta)
        * np.sin(phi),

        speed
        * np.sin(theta)
    ])

    # --------------------------------------------------------
    # Initial state
    # --------------------------------------------------------

    initial_state = np.array([
        SERVER_X,
        0.0,
        contact_height_m,

        initial_velocity[0],
        initial_velocity[1],
        initial_velocity[2]
    ])

    # --------------------------------------------------------
    # Integrate trajectory
    # --------------------------------------------------------

    solution = solve_ivp(
        lambda t, state:
            serve_trajectory(
                t,
                state,
                omega
            ),

        t_span=(0.0, 4.0),

        y0=initial_state,

        events=[
            net_event,
            ground_event
        ],

        rtol=1e-10,
        atol=1e-12,
        max_step=0.001,

        dense_output=True
    )

    return solution


print("=" * 70)
print("V8 VARIABLE-CONDITION SERVE SIMULATOR")
print("=" * 70)

print("\nFeatures:")
print("  ✓ Variable serve speed")
print("  ✓ Variable launch angle")
print("  ✓ Variable launch azimuth")
print("  ✓ Variable contact height")
print("  ✓ Variable spin rate")
print("  ✓ Correct rpm → rad/s conversion")
print("  ✓ Validated V7 aerodynamic model")
print("  ✓ Validated solve_ivp settings")

print("\nV8 Cell 5: SIMULATOR READY")
print("=" * 70)

In [ ]:
# V8 Cell 6 — V7 ↔ V8 Regression Test

# ------------------------------------------------------------
# Run the nominal zero-spin serve through both simulators
# ------------------------------------------------------------

v7_solution = simulate_serve_v7(
    speed_kmh=NOMINAL_SPEED_KMH,
    launch_angle_deg=NOMINAL_ANGLE_DEG,
    azimuth_deg=NOMINAL_AZIMUTH_DEG,
    omega=np.array([0.0, 0.0, 0.0])
)

v8_solution = simulate_serve_v8(
    speed_kmh=NOMINAL_SPEED_KMH,
    launch_angle_deg=NOMINAL_ANGLE_DEG,
    azimuth_deg=NOMINAL_AZIMUTH_DEG,
    contact_height_m=NOMINAL_CONTACT_HEIGHT_M,
    spin_rpm=NOMINAL_SPIN_RPM
)

# ------------------------------------------------------------
# Extract events
# ------------------------------------------------------------

v7_net, v7_landing = extract_serve_events(v7_solution)
v8_net, v8_landing = extract_serve_events(v8_solution)

# ------------------------------------------------------------
# Calculate differences
# ------------------------------------------------------------

net_difference = np.max(
    np.abs(v7_net - v8_net)
)

landing_difference = np.max(
    np.abs(v7_landing - v8_landing)
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 70)
print("V8 — V7 ↔ V8 REGRESSION TEST")
print("=" * 70)

print("\nV7 results:")
print(f"  Net:     {v7_net}")
print(f"  Landing: {v7_landing}")

print("\nV8 results:")
print(f"  Net:     {v8_net}")
print(f"  Landing: {v8_landing}")

print("\nMaximum differences:")
print(f"  Net state:     {net_difference:.3e}")
print(f"  Landing state: {landing_difference:.3e}")

regression_pass = (
    net_difference < 1e-10
    and landing_difference < 1e-10
)

print("\n" + "=" * 70)
print(
    f"V7 ↔ V8 REGRESSION: "
    f"{'PASS' if regression_pass else 'FAIL'}"
)
print("=" * 70)

In [ ]:
# V8 Cell 7 — Variable Condition Sanity Check

print("=" * 70)
print("V8 — VARIABLE CONDITION SANITY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Test 1: Contact height variation
# ------------------------------------------------------------

height_test = []

for height in [2.9, 3.0, 3.1]:

    solution = simulate_serve_v8(
        speed_kmh=NOMINAL_SPEED_KMH,
        launch_angle_deg=NOMINAL_ANGLE_DEG,
        azimuth_deg=NOMINAL_AZIMUTH_DEG,
        contact_height_m=height,
        spin_rpm=0.0
    )

    net_state, landing_state = extract_serve_events(
        solution
    )

    height_test.append({
        "contact_height_m": height,
        "net_z_m": net_state[2],
        "landing_x_m": landing_state[0]
    })

height_test_df = pd.DataFrame(height_test)

print("\nContact-height test:")
print(height_test_df.to_string(index=False))

# ------------------------------------------------------------
# Test 2: Spin variation
# ------------------------------------------------------------

spin_test = []

for spin in [-100.0, 0.0, 100.0]:

    solution = simulate_serve_v8(
        speed_kmh=NOMINAL_SPEED_KMH,
        launch_angle_deg=NOMINAL_ANGLE_DEG,
        azimuth_deg=NOMINAL_AZIMUTH_DEG,
        contact_height_m=NOMINAL_CONTACT_HEIGHT_M,
        spin_rpm=spin
    )

    net_state, landing_state = extract_serve_events(
        solution
    )

    clearance = net_clearance(
        net_state[2],
        net_state[1]
    )

    spin_test.append({
        "spin_rpm": spin,
        "net_clearance_m": clearance,
        "landing_x_m": landing_state[0],
        "landing_y_m": landing_state[1]
    })

spin_test_df = pd.DataFrame(spin_test)

print("\nSpin test:")
print(spin_test_df.to_string(index=False))

# ------------------------------------------------------------
# Basic sanity checks
# ------------------------------------------------------------

height_monotonic = (
    height_test_df["net_z_m"].is_monotonic_increasing
    and height_test_df["landing_x_m"].is_monotonic_increasing
)

spin_monotonic = (
    spin_test_df["net_clearance_m"].is_monotonic_decreasing
    and spin_test_df["landing_x_m"].is_monotonic_decreasing
)

print("\n" + "=" * 70)

print(
    f"Contact-height response: "
    f"{'PASS' if height_monotonic else 'CHECK'}"
)

print(
    f"Spin response: "
    f"{'PASS' if spin_monotonic else 'CHECK'}"
)

print("=" * 70)

In [ ]:
# V8 Cell 8A — Service Box Classification Helper

def is_inside_service_box(
    y,
    target_side="deuce"
):
    """
    Determine whether a landing position lies inside
    the selected singles service box.
    """

    if target_side == "deuce":
        return (
            0.0 <= y <= SERVICE_BOX_WIDTH
        )

    elif target_side == "ad":
        return (
            -SERVICE_BOX_WIDTH <= y <= 0.0
        )

    else:
        raise ValueError(
            "target_side must be 'deuce' or 'ad'"
        )


print("=" * 70)
print("V8 — SERVICE BOX CLASSIFIER")
print("=" * 70)

print("\nDeuce-box tests:")
print(
    f"y =  0.000 m → "
    f"{is_inside_service_box(0.0, 'deuce')}"
)
print(
    f"y =  2.000 m → "
    f"{is_inside_service_box(2.0, 'deuce')}"
)
print(
    f"y =  4.115 m → "
    f"{is_inside_service_box(4.115, 'deuce')}"
)
print(
    f"y =  4.200 m → "
    f"{is_inside_service_box(4.2, 'deuce')}"
)

print("\n" + "=" * 70)
print("V8 SERVICE BOX CLASSIFIER READY")
print("=" * 70)

In [ ]:
# V8 Cell 8 — Full-Factorial Robustness Experiment

print("=" * 70)
print("V8 — FULL-FACTORIAL ROBUSTNESS EXPERIMENT")
print("=" * 70)

results = []

total_cases = len(perturbation_df)

for i, row in perturbation_df.iterrows():

    solution = simulate_serve_v8(
        speed_kmh=row["speed_kmh"],
        launch_angle_deg=row["angle_deg"],
        azimuth_deg=row["azimuth_deg"],
        contact_height_m=row["contact_height_m"],
        spin_rpm=row["spin_rpm"]
    )

    net_state, landing_state = extract_serve_events(
        solution
    )

    # --------------------------------------------------------
    # Extract trajectory outcomes
    # --------------------------------------------------------

    if net_state is None:
        net_z = np.nan
        net_y = np.nan
        clearance = np.nan
        net_detected = False
    else:
        net_z = net_state[2]
        net_y = net_state[1]

        clearance = net_clearance(
            net_z,
            net_y
        )

        net_detected = True

    if landing_state is None:
        landing_x = np.nan
        landing_y = np.nan
        landing_detected = False
    else:
        landing_x = landing_state[0]
        landing_y = landing_state[1]

        landing_detected = True

    # --------------------------------------------------------
    # Legal serve classification
    # --------------------------------------------------------

    legal = (
        net_detected
        and landing_detected
        and clearance > 0.0
        and landing_x > NET_X
        and landing_x < SERVICE_LINE_X
        and is_inside_service_box(
            landing_y,
            TARGET_SIDE
        )
    )

    results.append({
        "speed_kmh": row["speed_kmh"],
        "angle_deg": row["angle_deg"],
        "azimuth_deg": row["azimuth_deg"],
        "contact_height_m": row["contact_height_m"],
        "spin_rpm": row["spin_rpm"],
        "net_clearance_m": clearance,
        "landing_x_m": landing_x,
        "landing_y_m": landing_y,
        "legal": legal
    })

    # --------------------------------------------------------
    # Progress indicator
    # --------------------------------------------------------

    if (i + 1) % 25 == 0 or (i + 1) == total_cases:
        print(
            f"Completed {i + 1} / {total_cases}"
        )

robustness_df = pd.DataFrame(results)

# ------------------------------------------------------------
# Robustness summary
# ------------------------------------------------------------

legal_count = int(
    robustness_df["legal"].sum()
)

total_count = len(
    robustness_df
)

robustness_fraction = (
    legal_count / total_count
)

print("\n" + "=" * 70)
print("ROBUSTNESS SUMMARY")
print("=" * 70)

print(f"Total perturbation cases: {total_count}")
print(f"Legal cases:              {legal_count}")
print(f"Illegal cases:            {total_count - legal_count}")
print(
    f"Full-factorial robustness fraction: "
    f"{robustness_fraction:.6f}"
)

print("\nFirst five results:")
print(
    robustness_df.head().to_string(
        index=False
    )
)

print("\n" + "=" * 70)
print("V8 FULL-FACTORIAL EXPERIMENT COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 9 — Failure Mode Classification

def classify_failure(row, target_side="deuce"):

    # --------------------------------------------------------
    # Net constraint
    # --------------------------------------------------------

    net_fault = (
        row["net_clearance_m"] <= 0.0
    )

    # --------------------------------------------------------
    # Depth constraints
    # --------------------------------------------------------

    long_fault = (
        row["landing_x_m"] >= SERVICE_LINE_X
    )

    short_fault = (
        row["landing_x_m"] <= NET_X
    )

    # --------------------------------------------------------
    # Lateral service-box constraint
    # --------------------------------------------------------

    if target_side == "deuce":
        wide_fault = (
            row["landing_y_m"] < 0.0
            or row["landing_y_m"] > SERVICE_BOX_WIDTH
        )

    else:
        wide_fault = (
            row["landing_y_m"] < -SERVICE_BOX_WIDTH
            or row["landing_y_m"] > 0.0
        )

    # --------------------------------------------------------
    # Classification
    # --------------------------------------------------------

    failures = []

    if net_fault:
        failures.append("NET")

    if long_fault:
        failures.append("LONG")

    if short_fault:
        failures.append("SHORT")

    if wide_fault:
        failures.append("WIDE")

    if len(failures) == 0:
        return "LEGAL"

    if len(failures) == 1:
        return failures[0]

    return "MULTIPLE"


robustness_df["failure_mode"] = robustness_df.apply(
    classify_failure,
    axis=1
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

failure_counts = (
    robustness_df["failure_mode"]
    .value_counts()
    .sort_index()
)

print("=" * 70)
print("V8 — FAILURE MODE CLASSIFICATION")
print("=" * 70)

print("\nFailure-mode counts:")

for mode, count in failure_counts.items():

    fraction = count / len(robustness_df)

    print(
        f"{mode:10s}: "
        f"{count:3d} "
        f"({fraction:.2%})"
    )

print("\n" + "=" * 70)

print(
    "V8 FAILURE CLASSIFICATION: "
    "COMPLETE"
)

print("=" * 70)

In [ ]:
# V8 Cell 10 — One-at-a-Time Robustness Sensitivity

def evaluate_single_parameter(
    parameter_name,
    values
):
    """
    Evaluate legality while varying one parameter and
    holding all other nominal conditions fixed.
    """

    rows = []

    for value in values:

        params = {
            "speed_kmh": NOMINAL_SPEED_KMH,
            "angle_deg": NOMINAL_ANGLE_DEG,
            "azimuth_deg": NOMINAL_AZIMUTH_DEG,
            "contact_height_m": NOMINAL_CONTACT_HEIGHT_M,
            "spin_rpm": NOMINAL_SPIN_RPM
        }

        params[parameter_name] = value

        solution = simulate_serve_v8(
            speed_kmh=params["speed_kmh"],
            launch_angle_deg=params["angle_deg"],
            azimuth_deg=params["azimuth_deg"],
            contact_height_m=params["contact_height_m"],
            spin_rpm=params["spin_rpm"]
        )

        net_state, landing_state = extract_serve_events(
            solution
        )

        clearance = net_clearance(
            net_state[2],
            net_state[1]
        )

        legal = (
            clearance > 0.0
            and landing_state[0] > NET_X
            and landing_state[0] < SERVICE_LINE_X
            and is_inside_service_box(
                landing_state[1],
                TARGET_SIDE
            )
        )

        rows.append({
            parameter_name: value,
            "net_clearance_m": clearance,
            "landing_x_m": landing_state[0],
            "landing_y_m": landing_state[1],
            "legal": legal
        })

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# Evaluate each variable across its V8 perturbation range
# ------------------------------------------------------------

sensitivity_results = {}

sensitivity_results["speed_kmh"] = evaluate_single_parameter(
    "speed_kmh",
    speed_levels
)

sensitivity_results["angle_deg"] = evaluate_single_parameter(
    "angle_deg",
    angle_levels
)

sensitivity_results["azimuth_deg"] = evaluate_single_parameter(
    "azimuth_deg",
    azimuth_levels
)

sensitivity_results["contact_height_m"] = evaluate_single_parameter(
    "contact_height_m",
    height_levels
)

sensitivity_results["spin_rpm"] = evaluate_single_parameter(
    "spin_rpm",
    spin_levels
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 70)
print("V8 — ONE-AT-A-TIME SENSITIVITY")
print("=" * 70)

for parameter, df in sensitivity_results.items():

    print(f"\n--- {parameter} ---")
    print(
        df.to_string(
            index=False
        )
    )

print("\n" + "=" * 70)
print("V8 ONE-AT-A-TIME SENSITIVITY COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 11 — Local Constraint Sensitivity

# ------------------------------------------------------------
# Extract nominal outcome
# ------------------------------------------------------------

nominal_df = robustness_df[
    (robustness_df["speed_kmh"] == NOMINAL_SPEED_KMH)
    & (robustness_df["angle_deg"] == NOMINAL_ANGLE_DEG)
    & (robustness_df["azimuth_deg"] == NOMINAL_AZIMUTH_DEG)
    & (robustness_df["contact_height_m"] == NOMINAL_CONTACT_HEIGHT_M)
    & (robustness_df["spin_rpm"] == NOMINAL_SPIN_RPM)
]

nominal_clearance = nominal_df["net_clearance_m"].iloc[0]
nominal_x = nominal_df["landing_x_m"].iloc[0]
nominal_y = nominal_df["landing_y_m"].iloc[0]

# ------------------------------------------------------------
# Central finite differences
# ------------------------------------------------------------

def central_sensitivity(
    parameter,
    low_value,
    high_value
):

    low_df = sensitivity_results[parameter]

    low_row = low_df[
        low_df[parameter] == low_value
    ].iloc[0]

    high_row = low_df[
        low_df[parameter] == high_value
    ].iloc[0]

    delta = high_value - low_value

    d_clearance = (
        high_row["net_clearance_m"]
        - low_row["net_clearance_m"]
    ) / delta

    d_x = (
        high_row["landing_x_m"]
        - low_row["landing_x_m"]
    ) / delta

    d_y = (
        high_row["landing_y_m"]
        - low_row["landing_y_m"]
    ) / delta

    return {
        "parameter": parameter,
        "d_clearance": d_clearance,
        "d_landing_x": d_x,
        "d_landing_y": d_y
    }


sensitivity_table = pd.DataFrame([

    central_sensitivity(
        "speed_kmh",
        speed_levels[0],
        speed_levels[-1]
    ),

    central_sensitivity(
        "angle_deg",
        angle_levels[0],
        angle_levels[-1]
    ),

    central_sensitivity(
        "azimuth_deg",
        azimuth_levels[0],
        azimuth_levels[-1]
    ),

    central_sensitivity(
        "contact_height_m",
        height_levels[0],
        height_levels[-1]
    ),

    central_sensitivity(
        "spin_rpm",
        spin_levels[0],
        spin_levels[-1]
    )
])

# ------------------------------------------------------------
# Normalize sensitivities by the actual perturbation range
# ------------------------------------------------------------

ranges = {
    "speed_kmh": SPEED_PERTURBATION_KMH,
    "angle_deg": ANGLE_PERTURBATION_DEG,
    "azimuth_deg": AZIMUTH_PERTURBATION_DEG,
    "contact_height_m": CONTACT_HEIGHT_PERTURBATION_M,
    "spin_rpm": SPIN_PERTURBATION_RPM
}

sensitivity_table["clearance_change_over_range"] = (
    sensitivity_table.apply(
        lambda row:
        abs(row["d_clearance"])
        * 2.0
        * ranges[row["parameter"]],
        axis=1
    )
)

sensitivity_table["landing_x_change_over_range"] = (
    sensitivity_table.apply(
        lambda row:
        abs(row["d_landing_x"])
        * 2.0
        * ranges[row["parameter"]],
        axis=1
    )
)

sensitivity_table["landing_y_change_over_range"] = (
    sensitivity_table.apply(
        lambda row:
        abs(row["d_landing_y"])
        * 2.0
        * ranges[row["parameter"]],
        axis=1
    )
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("V8 — LOCAL CONSTRAINT SENSITIVITY")
print("=" * 70)

print("\nCentral finite-difference sensitivities:")
print(
    sensitivity_table[
        [
            "parameter",
            "d_clearance",
            "d_landing_x",
            "d_landing_y"
        ]
    ].to_string(index=False)
)

print("\nAbsolute outcome change across the full perturbation range:")

print(
    sensitivity_table[
        [
            "parameter",
            "clearance_change_over_range",
            "landing_x_change_over_range",
            "landing_y_change_over_range"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("V8 LOCAL SENSITIVITY ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 12 — Launch Angle × Azimuth Robustness Map

# ------------------------------------------------------------
# Aggregate robustness over speed, contact height, and spin
# ------------------------------------------------------------

robustness_map = (
    robustness_df
    .groupby(
        ["angle_deg", "azimuth_deg"]
    )["legal"]
    .mean()
    .reset_index()
)

robustness_pivot = robustness_map.pivot(
    index="angle_deg",
    columns="azimuth_deg",
    values="legal"
)

print("=" * 70)
print("V8 — LAUNCH ANGLE × AZIMUTH ROBUSTNESS")
print("=" * 70)

print("\nRobustness fraction by launch angle and azimuth:")

print(
    robustness_pivot.to_string(
        float_format=lambda x: f"{x:.3f}"
    )
)

# ------------------------------------------------------------
# Heatmap
# ------------------------------------------------------------

plt.figure(figsize=(9, 6))

plt.imshow(
    robustness_pivot.values,
    aspect="auto",
    origin="lower",
    extent=[
        robustness_pivot.columns.min(),
        robustness_pivot.columns.max(),
        robustness_pivot.index.min(),
        robustness_pivot.index.max()
    ],
    vmin=0.0,
    vmax=1.0
)

plt.colorbar(
    label="Robustness fraction"
)

plt.xlabel("Launch azimuth (degrees)")
plt.ylabel("Launch angle (degrees)")

plt.title(
    "Robustness of the Serve Across Launch Angle and Azimuth"
)

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("V8 ROBUSTNESS MAP COMPLETE")
print("=" * 70)

# V8A — Interior Nominal Serve

The initial robustness experiment used a nominal azimuth of 0°, which places the trajectory on the centerline boundary of the deuce service box.

To avoid boundary-induced robustness bias, the primary V8 robustness analysis uses an interior nominal serve aimed at the geometric center of the deuce service box.

For the deuce box,

$$
0 \le y \le 4.115\ \mathrm{m},
$$

so the geometric center is

$$
y_{\mathrm{target}} = 2.0575\ \mathrm{m}.
$$

The nominal azimuth is determined numerically so that the modeled landing position is centered on this target.

The resulting serve is then used as the nominal state for the corrected V8 robustness experiment.

In [ ]:
# V8 Cell 13 — Determine Interior Nominal Azimuth

from scipy.optimize import brentq

# ------------------------------------------------------------
# Target: geometric center of the deuce service box
# ------------------------------------------------------------

TARGET_LANDING_Y_M = SERVICE_BOX_WIDTH / 2.0

# ------------------------------------------------------------
# Function whose root gives the desired azimuth
# ------------------------------------------------------------

def landing_y_error(azimuth_deg):

    solution = simulate_serve_v8(
        speed_kmh=NOMINAL_SPEED_KMH,
        launch_angle_deg=NOMINAL_ANGLE_DEG,
        azimuth_deg=azimuth_deg,
        contact_height_m=NOMINAL_CONTACT_HEIGHT_M,
        spin_rpm=NOMINAL_SPIN_RPM
    )

    _, landing_state = extract_serve_events(
        solution
    )

    if landing_state is None:
        raise RuntimeError(
            "Landing event was not detected."
        )

    return (
        landing_state[1]
        - TARGET_LANDING_Y_M
    )


# ------------------------------------------------------------
# Solve for the azimuth producing the box center
# ------------------------------------------------------------

INTERIOR_NOMINAL_AZIMUTH_DEG = brentq(
    landing_y_error,
    0.0,
    13.0,
    xtol=1e-10
)

# ------------------------------------------------------------
# Verify the solution
# ------------------------------------------------------------

verification_solution = simulate_serve_v8(
    speed_kmh=NOMINAL_SPEED_KMH,
    launch_angle_deg=NOMINAL_ANGLE_DEG,
    azimuth_deg=INTERIOR_NOMINAL_AZIMUTH_DEG,
    contact_height_m=NOMINAL_CONTACT_HEIGHT_M,
    spin_rpm=NOMINAL_SPIN_RPM
)

verification_net, verification_landing = (
    extract_serve_events(
        verification_solution
    )
)

verification_clearance = net_clearance(
    verification_net[2],
    verification_net[1]
)

azimuth_error = (
    verification_landing[1]
    - TARGET_LANDING_Y_M
)

nominal_interior_legal = (
    verification_clearance > 0.0
    and verification_landing[0] > NET_X
    and verification_landing[0] < SERVICE_LINE_X
    and is_inside_service_box(
        verification_landing[1],
        TARGET_SIDE
    )
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 70)
print("V8 — INTERIOR NOMINAL SERVE")
print("=" * 70)

print("\nTarget:")
print(
    f"Deuce-box center: "
    f"{TARGET_LANDING_Y_M:.6f} m"
)

print("\nSolved nominal azimuth:")
print(
    f"{INTERIOR_NOMINAL_AZIMUTH_DEG:.10f}°"
)

print("\nVerified trajectory:")

print(
    f"Net y:           "
    f"{verification_net[1]:.6f} m"
)

print(
    f"Net z:           "
    f"{verification_net[2]:.6f} m"
)

print(
    f"Net clearance:   "
    f"{verification_clearance:.6f} m"
)

print(
    f"Landing x:       "
    f"{verification_landing[0]:.6f} m"
)

print(
    f"Landing y:       "
    f"{verification_landing[1]:.6f} m"
)

print(
    f"Landing-y error: "
    f"{azimuth_error:+.3e} m"
)

print(
    f"\nNominal serve legal: "
    f"{nominal_interior_legal}"
)

print("\n" + "=" * 70)

if (
    abs(azimuth_error) < 1e-8
    and nominal_interior_legal
):
    print("V8 INTERIOR NOMINAL: PASS")
else:
    print("V8 INTERIOR NOMINAL: CHECK")

print("=" * 70)

In [ ]:
# V8 Cell 14 — Rebuild Perturbation Design Around Interior Nominal

# ------------------------------------------------------------
# Update the primary nominal azimuth
# ------------------------------------------------------------

NOMINAL_AZIMUTH_DEG = INTERIOR_NOMINAL_AZIMUTH_DEG

# ------------------------------------------------------------
# Rebuild the three-level perturbation grid
# ------------------------------------------------------------

speed_levels = np.array([
    NOMINAL_SPEED_KMH - SPEED_PERTURBATION_KMH,
    NOMINAL_SPEED_KMH,
    NOMINAL_SPEED_KMH + SPEED_PERTURBATION_KMH
])

angle_levels = np.array([
    NOMINAL_ANGLE_DEG - ANGLE_PERTURBATION_DEG,
    NOMINAL_ANGLE_DEG,
    NOMINAL_ANGLE_DEG + ANGLE_PERTURBATION_DEG
])

azimuth_levels = np.array([
    NOMINAL_AZIMUTH_DEG - AZIMUTH_PERTURBATION_DEG,
    NOMINAL_AZIMUTH_DEG,
    NOMINAL_AZIMUTH_DEG + AZIMUTH_PERTURBATION_DEG
])

height_levels = np.array([
    NOMINAL_CONTACT_HEIGHT_M - CONTACT_HEIGHT_PERTURBATION_M,
    NOMINAL_CONTACT_HEIGHT_M,
    NOMINAL_CONTACT_HEIGHT_M + CONTACT_HEIGHT_PERTURBATION_M
])

spin_levels = np.array([
    NOMINAL_SPIN_RPM - SPIN_PERTURBATION_RPM,
    NOMINAL_SPIN_RPM,
    NOMINAL_SPIN_RPM + SPIN_PERTURBATION_RPM
])

# ------------------------------------------------------------
# Full factorial design
# ------------------------------------------------------------

perturbation_cases = []

for speed in speed_levels:
    for angle in angle_levels:
        for azimuth in azimuth_levels:
            for height in height_levels:
                for spin in spin_levels:

                    perturbation_cases.append({
                        "speed_kmh": speed,
                        "angle_deg": angle,
                        "azimuth_deg": azimuth,
                        "contact_height_m": height,
                        "spin_rpm": spin
                    })

perturbation_df = pd.DataFrame(
    perturbation_cases
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

expected_cases = (
    len(speed_levels)
    * len(angle_levels)
    * len(azimuth_levels)
    * len(height_levels)
    * len(spin_levels)
)

design_pass = (
    len(perturbation_df)
    == expected_cases
)

print("=" * 70)
print("V8 — INTERIOR-NOMINAL PERTURBATION DESIGN")
print("=" * 70)

print("\nPrimary nominal:")
print(
    f"Speed:          "
    f"{NOMINAL_SPEED_KMH:.1f} km/h"
)

print(
    f"Launch angle:   "
    f"{NOMINAL_ANGLE_DEG:.6f}°"
)

print(
    f"Launch azimuth: "
    f"{NOMINAL_AZIMUTH_DEG:.10f}°"
)

print(
    f"Contact height: "
    f"{NOMINAL_CONTACT_HEIGHT_M:.2f} m"
)

print(
    f"Spin:           "
    f"{NOMINAL_SPIN_RPM:.0f} rpm"
)

print("\nPerturbation levels:")
print(
    f"Speed:          {speed_levels}"
)
print(
    f"Launch angle:   {angle_levels}"
)
print(
    f"Launch azimuth: {azimuth_levels}"
)
print(
    f"Contact height: {height_levels}"
)
print(
    f"Spin:            {spin_levels}"
)

print("\nExpected cases:", expected_cases)
print("Actual cases:  ", len(perturbation_df))

print("\n" + "=" * 70)
print(
    f"V8 INTERIOR PERTURBATION DESIGN: "
    f"{'PASS' if design_pass else 'FAIL'}"
)
print("=" * 70)

In [ ]:
# V8 Cell 15 — Corrected Full-Factorial Robustness Experiment

print("=" * 70)
print("V8 — CORRECTED FULL-FACTORIAL ROBUSTNESS EXPERIMENT")
print("=" * 70)

results = []

total_cases = len(perturbation_df)

for i, row in perturbation_df.iterrows():

    solution = simulate_serve_v8(
        speed_kmh=row["speed_kmh"],
        launch_angle_deg=row["angle_deg"],
        azimuth_deg=row["azimuth_deg"],
        contact_height_m=row["contact_height_m"],
        spin_rpm=row["spin_rpm"]
    )

    net_state, landing_state = extract_serve_events(
        solution
    )

    # --------------------------------------------------------
    # Extract trajectory outcomes
    # --------------------------------------------------------

    if net_state is None:
        net_z = np.nan
        net_y = np.nan
        clearance = np.nan
        net_detected = False
    else:
        net_z = net_state[2]
        net_y = net_state[1]

        clearance = net_clearance(
            net_z,
            net_y
        )

        net_detected = True

    if landing_state is None:
        landing_x = np.nan
        landing_y = np.nan
        landing_detected = False
    else:
        landing_x = landing_state[0]
        landing_y = landing_state[1]

        landing_detected = True

    # --------------------------------------------------------
    # Legal serve classification
    # --------------------------------------------------------

    legal = (
        net_detected
        and landing_detected
        and clearance > 0.0
        and landing_x > NET_X
        and landing_x < SERVICE_LINE_X
        and is_inside_service_box(
            landing_y,
            TARGET_SIDE
        )
    )

    results.append({
        "speed_kmh": row["speed_kmh"],
        "angle_deg": row["angle_deg"],
        "azimuth_deg": row["azimuth_deg"],
        "contact_height_m": row["contact_height_m"],
        "spin_rpm": row["spin_rpm"],
        "net_clearance_m": clearance,
        "landing_x_m": landing_x,
        "landing_y_m": landing_y,
        "legal": legal
    })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 25 == 0 or (i + 1) == total_cases:
        print(
            f"Completed {i + 1} / {total_cases}"
        )

# ------------------------------------------------------------
# Create final dataframe
# ------------------------------------------------------------

robustness_df = pd.DataFrame(results)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

legal_count = int(
    robustness_df["legal"].sum()
)

total_count = len(
    robustness_df
)

illegal_count = (
    total_count
    - legal_count
)

robustness_fraction = (
    legal_count / total_count
)

print("\n" + "=" * 70)
print("CORRECTED ROBUSTNESS SUMMARY")
print("=" * 70)

print(
    f"Total perturbation cases: "
    f"{total_count}"
)

print(
    f"Legal cases:              "
    f"{legal_count}"
)

print(
    f"Illegal cases:            "
    f"{illegal_count}"
)

print(
    f"Full-factorial robustness fraction: "
    f"{robustness_fraction:.6f}"
)

print(
    f"Full-factorial robustness percentage: "
    f"{robustness_fraction:.2%}"
)

print("\nFirst five results:")

print(
    robustness_df.head().to_string(
        index=False
    )
)

print("\n" + "=" * 70)
print("V8 CORRECTED FULL-FACTORIAL EXPERIMENT COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 16 — Corrected Failure Mode Classification

# ------------------------------------------------------------
# Classify each corrected robustness case
# ------------------------------------------------------------

robustness_df["failure_mode"] = robustness_df.apply(
    classify_failure,
    axis=1
)

failure_counts = (
    robustness_df["failure_mode"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Calculate percentages
# ------------------------------------------------------------

failure_summary = []

for mode, count in failure_counts.items():

    failure_summary.append({
        "failure_mode": mode,
        "count": int(count),
        "percent_all_cases": (
            100.0 * count / len(robustness_df)
        ),
        "percent_illegal_cases": (
            100.0 * count / (
                robustness_df["legal"].eq(False).sum()
            )
            if mode != "LEGAL"
            else np.nan
        )
    })

failure_summary_df = pd.DataFrame(
    failure_summary
)

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 70)
print("V8 — CORRECTED FAILURE MODE ANALYSIS")
print("=" * 70)

print(
    failure_summary_df.to_string(
        index=False,
        formatters={
            "percent_all_cases":
                lambda x: f"{x:.2f}%",
            "percent_illegal_cases":
                lambda x:
                    "—"
                    if pd.isna(x)
                    else f"{x:.2f}%"
        }
    )
)

print("\n" + "=" * 70)
print("V8 CORRECTED FAILURE ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 17 — Verify Corrected Failure Mechanisms

illegal_df = robustness_df[
    ~robustness_df["legal"]
].copy()

# ------------------------------------------------------------
# Independent constraint checks
# ------------------------------------------------------------

illegal_df["net_ok"] = (
    illegal_df["net_clearance_m"] > 0.0
)

illegal_df["depth_ok"] = (
    (illegal_df["landing_x_m"] > NET_X)
    & (illegal_df["landing_x_m"] < SERVICE_LINE_X)
)

illegal_df["lateral_ok"] = (
    illegal_df["landing_y_m"] >= 0.0
) & (
    illegal_df["landing_y_m"] <= SERVICE_BOX_WIDTH
)

print("=" * 70)
print("V8 — INDEPENDENT FAILURE MECHANISM VERIFICATION")
print("=" * 70)

print("\nIllegal cases:", len(illegal_df))

print(
    f"Net constraint satisfied:     "
    f"{illegal_df['net_ok'].sum()} / {len(illegal_df)}"
)

print(
    f"Depth constraint satisfied:   "
    f"{illegal_df['depth_ok'].sum()} / {len(illegal_df)}"
)

print(
    f"Lateral constraint satisfied: "
    f"{illegal_df['lateral_ok'].sum()} / {len(illegal_df)}"
)

print("\nConstraint violations among illegal cases:")

print(
    f"Net faults:     "
    f"{(~illegal_df['net_ok']).sum()}"
)

print(
    f"Depth faults:   "
    f"{(~illegal_df['depth_ok']).sum()}"
)

print(
    f"Lateral faults: "
    f"{(~illegal_df['lateral_ok']).sum()}"
)

# ------------------------------------------------------------
# Strong verification condition:
# all illegal cases fail net while satisfying depth/lateral.
# ------------------------------------------------------------

failure_mechanism_pass = (
    (~illegal_df["net_ok"]).all()
    and illegal_df["depth_ok"].all()
    and illegal_df["lateral_ok"].all()
)

print("\n" + "=" * 70)

print(
    f"NET-DOMINATED FAILURE VERIFICATION: "
    f"{'PASS' if failure_mechanism_pass else 'CHECK'}"
)

print("=" * 70)

In [ ]:
# V8 Cell 18 — Robustness Landscape: Launch Angle × Azimuth

# ------------------------------------------------------------
# Aggregate robustness over:
#   speed perturbation
#   contact-height perturbation
#   spin perturbation
#
# while displaying robustness as a function of:
#   launch angle × azimuth
# ------------------------------------------------------------

robustness_map = (
    robustness_df
    .groupby(["angle_deg", "azimuth_deg"])["legal"]
    .mean()
    .reset_index()
)

robustness_pivot = robustness_map.pivot(
    index="angle_deg",
    columns="azimuth_deg",
    values="legal"
)

print("=" * 70)
print("V8 — ROBUSTNESS LANDSCAPE")
print("=" * 70)

print("\nFraction of perturbation cases remaining legal:")
print(
    robustness_pivot
    .round(3)
    .to_string()
)

print("\n" + "=" * 70)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(
    robustness_pivot.values,
    origin="lower",
    aspect="auto",
    extent=[
        robustness_pivot.columns.min(),
        robustness_pivot.columns.max(),
        robustness_pivot.index.min(),
        robustness_pivot.index.max()
    ],
    vmin=0.0,
    vmax=1.0
)

ax.set_xlabel("Azimuth angle (degrees)")
ax.set_ylabel("Launch angle (degrees)")
ax.set_title(
    "Robustness Landscape: Fraction of Perturbed Serves Remaining Legal"
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Legal fraction")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nRobustness map dimensions:")
print(f"Rows:    {robustness_pivot.shape[0]}")
print(f"Columns: {robustness_pivot.shape[1]}")

print("\n" + "=" * 70)
print("V8 ROBUSTNESS LANDSCAPE COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 19 — Worst-Case Net Clearance Landscape

# ------------------------------------------------------------
# For each launch-angle × azimuth combination, determine
# the minimum net clearance across all speed, height, and
# spin perturbations.
# ------------------------------------------------------------

clearance_map = (
    robustness_df
    .groupby(["angle_deg", "azimuth_deg"])["net_clearance_m"]
    .min()
    .reset_index()
)

clearance_pivot = clearance_map.pivot(
    index="angle_deg",
    columns="azimuth_deg",
    values="net_clearance_m"
)

print("=" * 70)
print("V8 — WORST-CASE NET CLEARANCE LANDSCAPE")
print("=" * 70)

print("\nMinimum net clearance across perturbation cases (m):")

print(
    clearance_pivot
    .round(4)
    .to_string()
)

print("\nWorst overall net clearance:")
print(
    f"{robustness_df['net_clearance_m'].min():.6f} m"
)

print("Best overall net clearance:")
print(
    f"{robustness_df['net_clearance_m'].max():.6f} m"
)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(
    clearance_pivot.values,
    origin="lower",
    aspect="auto",
    extent=[
        clearance_pivot.columns.min(),
        clearance_pivot.columns.max(),
        clearance_pivot.index.min(),
        clearance_pivot.index.max()
    ]
)

ax.set_xlabel("Azimuth angle (degrees)")
ax.set_ylabel("Launch angle (degrees)")
ax.set_title(
    "Worst-Case Net Clearance Across Perturbations"
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Minimum net clearance (m)")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("\nClearance map dimensions:")
print(f"Rows:    {clearance_pivot.shape[0]}")
print(f"Columns: {clearance_pivot.shape[1]}")

print("\n" + "=" * 70)
print("V8 WORST-CASE CLEARANCE LANDSCAPE COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 20 — Empirical Uncertainty Contribution to Net Clearance

print("=" * 70)
print("V8 — EMPIRICAL UNCERTAINTY CONTRIBUTION")
print("=" * 70)

# ------------------------------------------------------------
# Calculate the range of net clearance associated with each
# uncertainty variable while averaging over the other variables.
# ------------------------------------------------------------

variables = {
    "speed_kmh": "Speed (km/h)",
    "angle_deg": "Launch angle (deg)",
    "azimuth_deg": "Azimuth (deg)",
    "contact_height_m": "Contact height (m)",
    "spin_rpm": "Spin (rpm)"
}

contribution_rows = []

for column, label in variables.items():

    grouped = (
        robustness_df
        .groupby(column)["net_clearance_m"]
        .mean()
    )

    contribution_rows.append({
        "variable": label,
        "min_group_mean_clearance_m": grouped.min(),
        "max_group_mean_clearance_m": grouped.max(),
        "range_m": grouped.max() - grouped.min()
    })

contribution_df = pd.DataFrame(contribution_rows)

# Sort by largest empirical effect
contribution_df = contribution_df.sort_values(
    "range_m",
    ascending=False
).reset_index(drop=True)

print("\nMean net clearance across levels of each uncertainty variable:")
print(
    contribution_df.to_string(
        index=False,
        formatters={
            "min_group_mean_clearance_m": lambda x: f"{x:.6f}",
            "max_group_mean_clearance_m": lambda x: f"{x:.6f}",
            "range_m": lambda x: f"{x:.6f}"
        }
    )
)

print("\n" + "=" * 70)

# ------------------------------------------------------------
# Identify dominant variable
# ------------------------------------------------------------

dominant_variable = contribution_df.iloc[0]["variable"]
dominant_range = contribution_df.iloc[0]["range_m"]

print(
    f"Dominant empirical factor: {dominant_variable}"
)

print(
    f"Range in mean net clearance: {dominant_range:.6f} m"
)

print("\n" + "=" * 70)
print("V8 EMPIRICAL UNCERTAINTY ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 21 — Marginal Legal Fraction by Perturbation Variable

print("=" * 70)
print("V8 — MARGINAL ROBUSTNESS BY PERTURBATION VARIABLE")
print("=" * 70)

variables = [
    ("speed_kmh", "Speed (km/h)"),
    ("angle_deg", "Launch angle (deg)"),
    ("azimuth_deg", "Azimuth (deg)"),
    ("contact_height_m", "Contact height (m)"),
    ("spin_rpm", "Spin (rpm)")
]

marginal_tables = {}

for column, label in variables:

    marginal = (
        robustness_df
        .groupby(column)["legal"]
        .agg(
            legal_cases="sum",
            total_cases="count",
            legal_fraction="mean"
        )
        .reset_index()
    )

    marginal["legal_percent"] = (
        100.0 * marginal["legal_fraction"]
    )

    marginal_tables[column] = marginal

    print(f"\n{label}")
    print("-" * 70)

    print(
        marginal[
            [column, "legal_cases", "total_cases", "legal_percent"]
        ].to_string(
            index=False,
            formatters={
                "legal_percent": lambda x: f"{x:.2f}%"
            }
        )
    )

# ------------------------------------------------------------
# Identify the strongest marginal change in legal fraction
# ------------------------------------------------------------

marginal_effects = []

for column, label in variables:

    values = marginal_tables[column]["legal_percent"]

    marginal_effects.append({
        "variable": label,
        "min_legal_percent": values.min(),
        "max_legal_percent": values.max(),
        "range_percentage_points": values.max() - values.min()
    })

marginal_effects_df = (
    pd.DataFrame(marginal_effects)
    .sort_values(
        "range_percentage_points",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("MARGINAL ROBUSTNESS EFFECT")
print("=" * 70)

print(
    marginal_effects_df.to_string(
        index=False,
        formatters={
            "min_legal_percent": lambda x: f"{x:.2f}%",
            "max_legal_percent": lambda x: f"{x:.2f}%",
            "range_percentage_points": lambda x: f"{x:.2f}"
        }
    )
)

print("\n" + "=" * 70)
print("V8 MARGINAL ROBUSTNESS ANALYSIS COMPLETE")
print("=" * 70)

In [ ]:
# V8 Cell 22 — Final V8 Results Summary

print("=" * 70)
print("V8 — ROBUSTNESS & UNCERTAINTY FINAL SUMMARY")
print("=" * 70)

print("\nNOMINAL SERVE")
print(f"Speed:              {NOMINAL_SPEED_KMH:.1f} km/h")
print(f"Launch angle:       {NOMINAL_ANGLE_DEG:.6f}°")
print(f"Azimuth:            {INTERIOR_NOMINAL_AZIMUTH_DEG:.6f}°")
print(f"Contact height:     {NOMINAL_CONTACT_HEIGHT_M:.2f} m")
print(f"Spin:               {NOMINAL_SPIN_RPM:.1f} rpm")

print("\nPERTURBATION GRID")
print(f"Total cases:        {len(robustness_df)}")
print(f"Legal cases:        {robustness_df['legal'].sum()}")
print(f"Illegal cases:      {(~robustness_df['legal']).sum()}")
print(
    f"Legal fraction:     "
    f"{100.0 * robustness_df['legal'].mean():.2f}%"
)

print("\nFAILURE MODES")
for mode, count in failure_counts.items():
    print(
        f"{mode:18s} "
        f"{int(count):3d} cases "
        f"({100.0 * count / len(robustness_df):.2f}%)"
    )

print("\nNET CLEARANCE")
print(
    f"Minimum:            "
    f"{robustness_df['net_clearance_m'].min():.6f} m"
)
print(
    f"Maximum:            "
    f"{robustness_df['net_clearance_m'].max():.6f} m"
)

print("\nDOMINANT EMPIRICAL PERTURBATION")
print(f"Factor:             {dominant_variable}")
print(f"Clearance range:    {dominant_range:.6f} m")

print("\n" + "=" * 70)
print("V8 FINAL SUMMARY COMPLETE")
print("=" * 70)